In [47]:
import pandas as pd

In [48]:
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

print(train.shape, test.shape)
train.info()

(891, 12) (418, 11)
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [49]:
train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [50]:
survival_rate = train['Survived'].mean()
survival_rate

np.float64(0.3838383838383838)

In [51]:
survival_rate_gender = train.groupby('Sex')['Survived'].mean()
survival_rate_gender

Sex
female    0.742038
male      0.188908
Name: Survived, dtype: float64

In [52]:
survivors = train[train['Survived'] == 1]
survivors_female = (survivors['Sex'] == 'female').mean()
print(survivors_female)

0.6812865497076024


In [53]:
survival_rate_pclass = train.groupby('Pclass')['Survived'].mean()
survival_rate_pclass

Pclass
1    0.629630
2    0.472826
3    0.242363
Name: Survived, dtype: float64

Conclusion:
* 38% of all people aboard lived, meaning a model who predicts with 0.62 accuracy didn't learn anything
* 74% of females and 18% of males lived, meaning it was mostly women who lived; further investigation: out of all survivors, 68% are women, so a model who only predicts based on gender will get a pretty high result
* Pclass matters a lot, 62% of people who had been in first class survived, and the percentage drops massively as the class goes down to third


In [54]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [55]:
test.info()

<class 'pandas.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Pclass       418 non-null    int64  
 2   Name         418 non-null    str    
 3   Sex          418 non-null    str    
 4   Age          332 non-null    float64
 5   SibSp        418 non-null    int64  
 6   Parch        418 non-null    int64  
 7   Ticket       418 non-null    str    
 8   Fare         417 non-null    float64
 9   Cabin        91 non-null     str    
 10  Embarked     418 non-null    str    
dtypes: float64(2), int64(4), str(5)
memory usage: 36.1 KB


Feature Engineering

In [ ]:
def get_title(df: pd.DataFrame) -> pd.DataFrame:
	df = df.copy()
	df['Title'] = df['Name'].str.extract(r',\s*([^.]+)\.', expand=False)
	df['Title'] = df['Title'].replace({'Mme': 'Mrs', 'Mlle': 'Miss', 'Ms': 'Miss'})

	# replaces every title other than the ones defined in common with 'Rare'
	common = {'Miss', 'Mrs', 'Mr', 'Master'}
	df['Title'] = df['Title'].where(df['Title'].isin(common), 'Rare')

	return df

def fit_age_medians(df: pd.DataFrame) -> pd.DataFrame:
	return df.groupby('Title')['Age'].median()

def fill_age(df: pd.DataFrame, age_medians: pd.Series) -> pd.DataFrame:
	df = df.copy()
	df['AgeMissing'] = df['Age'].isna().astype(int)
	df['Age'] = df['Age'].fillna(df['Title'].map(age_medians))
	return df

In [57]:
train = get_title(train)
test = get_title(test)

age_medians = fit_age_medians(train)
print(age_medians)

train = fill_age(train, age_medians)
test = fill_age(test, age_medians)

print(train['Age'].isna().sum(), test['Age'].isna().sum())

Title
Master     3.5
Miss      21.0
Mr        30.0
Mrs       35.0
Rare      48.5
Name: Age, dtype: float64
0 0


In [ ]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
	df = df.copy()
	df['HasCabin'] = df['Cabin'].notna().astype(int)
	df['FamilySize'] = df['Parch'] + df['SibSp'] + 1
	df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
	return df

train = add_features(train)
test = add_features(test)


In [59]:
all_tickets = pd.concat([train['Ticket'], test['Ticket']])
ticket_counts = all_tickets.value_counts()
print(ticket_counts.head(10))

Ticket
CA. 2343        11
CA 2144          8
1601             8
347082           7
347077           7
3101295          7
S.O.C. 14879     7
PC 17608         7
382652           6
19950            6
Name: count, dtype: int64


In [ ]:
def add_ticket_features(df: pd.DataFrame, ticket_counts: pd.Series) -> pd.DataFrame:
	df = df.copy()
	df['TicketFreq'] = df['Ticket'].map(ticket_counts)
	df['FarePerPerson'] = df['Fare'] / df['TicketFreq']
	return df


train = add_ticket_features(train, ticket_counts)
test = add_ticket_features(test, ticket_counts)

In [61]:
test_ids = test['PassengerId']

columns = ['PassengerId', 'Name', 'Ticket', 'Cabin']
train = train.drop(columns=columns)
test = test.drop(columns=columns)

In [66]:
from sklearn.preprocessing import OneHotEncoder

cat_cols = ['Embarked', 'Sex', 'Title']

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False).set_output(transform='pandas')
encoder.fit(train[cat_cols])

def encode(df: pd.DataFrame, encoder: OneHotEncoder) -> pd.DataFrame:
	encoded = encoder.transform(df[cat_cols])
	return pd.concat([df.drop(columns=cat_cols), encoded], axis = 1)

train = encode(train, encoder)
test = encode(test, encoder)

In [67]:
print(train.shape, test.shape)
print(train.dtypes)
print(list(train.columns))
print(list(test.columns))

(891, 23) (418, 22)
Survived           int64
Pclass             int64
Age              float64
SibSp              int64
Parch              int64
Fare             float64
AgeMissing         int64
HasCabin           int64
FamilySize         int64
IsAlone            int64
TicketFreq         int64
FarePerPerson    float64
Embarked_C       float64
Embarked_Q       float64
Embarked_S       float64
Embarked_nan     float64
Sex_female       float64
Sex_male         float64
Title_Master     float64
Title_Miss       float64
Title_Mr         float64
Title_Mrs        float64
Title_Rare       float64
dtype: object
['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'AgeMissing', 'HasCabin', 'FamilySize', 'IsAlone', 'TicketFreq', 'FarePerPerson', 'Embarked_C', 'Embarked_Q', 'Embarked_S', 'Embarked_nan', 'Sex_female', 'Sex_male', 'Title_Master', 'Title_Miss', 'Title_Mr', 'Title_Mrs', 'Title_Rare']
['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'AgeMissing', 'HasCabin', 'FamilySize', 'IsAlone', 'Tick

In [70]:
X = train.drop(columns=['Survived'])
y = train['Survived']

In [75]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=67)
model = RandomForestClassifier(n_estimators=300, max_depth=5, random_state=67)

# As i've mentioned earlier, a model relying only on sex will have a decent accuracy
basic = cross_val_score(model, X[['Sex_female']], y, cv=cv)
print(basic.mean())

scores = cross_val_score(model, X, y, cv=cv)
print(scores.mean())

0.7867491055175445
0.8338899001945892


In [76]:
model.fit(X, y)

test = test[X.columns]
prediction = model.predict(test)

submission = pd.DataFrame({
	'PassengerId': test_ids,
	'Survived': prediction,
})

submission.to_csv('data/submission.csv', index=False)